# PRAGma App — LOT 기반 에칭 공정 분석 시스템

**파이프라인:**
1. LOT 번호 입력 → MES(CSV Mock)에서 공정 데이터 자동 조회
2. LightGBM 모델로 에칭 속도 예측
3. **EXAONE Model RAG** (LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct + Hybrid BM25+Vector + ML 예측/OPLS 컨텍스트 주입)으로 공정 해석 및 조치 방안 제시

**Google Colab 실행 필수** — EXAONE은 GPU(4bit 양자화)가 필요합니다.  
**Secrets에 `HF_TOKEN` 필요** — HuggingFace 인증용

## 0. 환경 설정

In [ ]:
# 1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 2. GitHub에서 코드 가져오기
import subprocess
result = subprocess.run(["git", "clone", "https://github.com/heyitsmialee/PRAGma.git"],
                        capture_output=True, text=True)
if "already exists" in result.stderr:
    print("PRAGma 이미 존재 — pull로 최신화")
    subprocess.run(["git", "-C", "/content/PRAGma", "pull"], check=True)
else:
    print(result.stdout or result.stderr)

import os
os.chdir("/content/PRAGma")
print(f"작업 디렉토리: {os.getcwd()}")

In [ ]:
import os
import json
import pickle
import subprocess
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# ── 패키지 설치 ───────────────────────────────────────────────────────────────
PACKAGES = [
    "langchain", "langchain-core", "langchain-community",
    "langchain-text-splitters", "langchain-huggingface",
    "langchain-chroma", "rank_bm25",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U"] + PACKAGES, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.52.0", "accelerate", "bitsandbytes", "torch"], check=True)
print("패키지 설치 완료")

# ── HuggingFace 토큰 (Colab Secrets) ─────────────────────────────────────────
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Secrets에 HF_TOKEN이 없습니다. 왼쪽 🔑 → Add new secret 으로 추가하세요.")

from huggingface_hub import login
login(token=HF_TOKEN)
print("HuggingFace 로그인 완료")

# ── 경로 설정 ─────────────────────────────────────────────────────────────────
# GitHub clone 경로 우선, Drive fallback
GITHUB_DIR = Path("/content/PRAGma")
DRIVE_DIR  = Path("/content/drive/MyDrive/Colab Notebooks/PRAGma")

MODEL_PATH      = GITHUB_DIR / "notebooks/best_LightGBM_mass_speed_regressor.pkl"
CSV_PATH        = GITHUB_DIR / "data/Train_0319.csv"
PAPER_JSON_PATH = GITHUB_DIR / "data/rag_data_all.json"
SHAP_MD_PATH    = GITHUB_DIR / "notebooks/shap_analysis_for_rag.md"
CHROMA_DIR      = "/content/pragma_chroma"

# Drive fallback (GitHub에 없는 경우)
if not MODEL_PATH.exists():
    MODEL_PATH = DRIVE_DIR / "best_LightGBM_mass_speed_regressor.pkl"
if not CSV_PATH.exists():
    CSV_PATH   = DRIVE_DIR / "Train_0319.csv"

print(f"모델 경로  : {MODEL_PATH}")
print(f"CSV 경로   : {CSV_PATH}")
print(f"JSON 경로  : {PAPER_JSON_PATH}")
print(f"SHAP MD    : {SHAP_MD_PATH}")
print("설정 완료")

## 1. 모델 및 데이터 로드

In [ ]:
# LightGBM 모델 로드
with open(MODEL_PATH, "rb") as f:
    lgbm_model = pickle.load(f)

MODEL_FEATURES = lgbm_model.feature_name_
print(f"모델 로드 완료 — 피처 수: {len(MODEL_FEATURES)}")
print(f"피처 목록: {MODEL_FEATURES[:8]} ...")

# MES Mock 데이터 로드 (실제 환경에서는 MES API 호출로 대체)
df_mes = pd.read_csv(CSV_PATH, encoding="cp949")
print(f"\nMES Mock 데이터 로드 완료 — LOT 수: {len(df_mes)}, 컬럼 수: {len(df_mes.columns)}")
print(f"LOT 예시: {df_mes['LOT'].head(5).tolist()}")

# RAG 지식베이스 로드
with open(PAPER_JSON_PATH, "r", encoding="utf-8") as f:
    paper_rules = json.load(f)

shap_content = SHAP_MD_PATH.read_text(encoding="utf-8") if SHAP_MD_PATH.exists() else ""

print(f"\nRAG 지식베이스 로드 완료")
print(f"  논문 rule 수: {len(paper_rules)}")
print(f"  SHAP 분석 MD 존재: {bool(shap_content)}")

## 2. CSV 컬럼 → 모델 피처 매핑

실제 MES에서 받는 raw 컬럼명을 LightGBM 모델 입력 피처명으로 변환합니다.

In [ ]:
# ── 연속형 컬럼 매핑 (동적 탐색) ─────────────────────────────────────────────
CONTINUOUS_MAP = {
    "Cu 표면두께 Max_Val"   : "cu_thick_max",
    "Cu 표면두께 AVG_VAL"   : "cu_thick_avg",
    "Cu 표면두께 Min_Val"   : "cu_thick_min",
    "Cu 표면두께 Std_Val"   : "cu_thick_std",
    "Cu 표면두께 Median_Val": "cu_thick_median",
}

_feat_suffix_map = {
    "Etch factor"             : "etch_factor",
    "Etching(염화동) - Cu"    : "meas_etch_cu",
    "Etching(염화동) - HCl"   : "meas_etch_hcl",
    "Etching(염화동) - 비중"  : "meas_etch_sg",
    "Etching(염화동) - 온도"  : "meas_etch_temp",
    "Etching-첨가제(HB-120EF)": "meas_etch_additive",
    "Etching량"               : "meas_etch_amount",
    "Soft Etch - Cu"          : "meas_softetch_cu",
    "Soft Etch - H2SO4"       : "meas_softetch_h2so4",
    "Soft Etch - SPS"         : "meas_softetch_sps",
    "박리액 - 농도"           : "meas_strip_conc",
    "수세수 - pH"             : "meas_rinse_ph",
    "현상액 - pH"             : "meas_dev_ph",
    "현상액 - 농도"           : "meas_dev_conc",
}

for col in df_mes.columns:
    if "분석치" in col:
        suffix = col.split("_", 1)[-1] if "_" in col else col
        if suffix in _feat_suffix_map:
            CONTINUOUS_MAP[col] = _feat_suffix_map[suffix]

CONTINUOUS_MAP_RESOLVED = CONTINUOUS_MAP

print(f"매핑 완료: {len(CONTINUOUS_MAP_RESOLVED)}개 연속형 피처")
for k, v in CONTINUOUS_MAP_RESOLVED.items():
    print(f"  {k!r:45s} → {v}")

## 3. MES Mock 함수

실제 환경에서는 MES REST API 호출로 교체합니다.

In [ ]:
def get_lot_data(lotno: str) -> dict | None:
    """MES Mock: LOT 번호 → 공정 데이터 딕셔너리 반환
    
    실제 MES 연동 시 이 함수만 교체하면 됩니다:
        response = requests.get(f"{MES_URL}/lot/{lotno}")
        return response.json()
    """
    rows = df_mes[df_mes["LOT"] == lotno]
    if rows.empty:
        print(f"[경고] LOT '{lotno}' 를 찾을 수 없습니다.")
        print(f"  사용 가능한 LOT 예시: {df_mes['LOT'].head(10).tolist()}")
        return None
    return rows.iloc[0].to_dict()


# 테스트
sample = get_lot_data("A20000")
if sample:
    print(f"LOT A20000 조회 성공")
    print(f"  실제 에칭 속도: {sample.get('부식 Speed')} m/min")
    print(f"  에칭 온도: {sample.get('분析치_Etching(염화동) - 온도')} °C")
    print(f"  에칭 비중: {sample.get('분析치_Etching(염화동) - 비중')}")

## 4. 피처 변환 (MES 데이터 → LightGBM 입력)

In [ ]:
# ── 범주형 컬럼 정의 ───────────────────────────────────────────────────────────
CATEGORICAL_COLS = {
    "재작업사유" : {
        "prefix": "rework_history",
        "values": ["Unknown", "기타", "기판 겹침", "두께 미달", "딤플", "설비 에러"],
        "sep"   : "_",   # 모델은 공백을 언더스코어로 저장했음
    },
    "노광 설비정보": {
        "prefix": "expo_eq_id",
        "values": [f"EXP-{i:03d}" for i in range(1, 8)],
        "sep"   : "_",
    },
    "DES 설비정보": {
        "prefix": "des_eq_id",
        "values": [f"DES-{i:03d}" for i in range(1, 7)],
        "sep"   : "_",
    },
    "정면 설비정보": {
        "prefix": "brush_eq_id",
        "values": [f"PRE-{i:03d}" for i in range(1, 8)],
        "sep"   : "_",
    },
}


def prepare_features(lot_data: dict) -> pd.DataFrame:
    """MES 딕셔너리 → LightGBM 입력 DataFrame (1행)"""
    row = {}

    # 1. 연속형 피처
    for csv_col, feat_name in CONTINUOUS_MAP_RESOLVED.items():
        val = lot_data.get(csv_col, np.nan)
        row[feat_name] = float(val) if val is not None and str(val) not in ("", "nan", "None") else np.nan

    # 2. 범주형 → one-hot
    for csv_col, cfg in CATEGORICAL_COLS.items():
        raw_val = str(lot_data.get(csv_col, "")).strip()
        # 공백을 언더스코어로 변환 (rework_history 한정)
        norm_val = raw_val.replace(" ", "_") if cfg["prefix"] == "rework_history" else raw_val
        for v in cfg["values"]:
            norm_v = v.replace(" ", "_") if cfg["prefix"] == "rework_history" else v
            feat_key = f"{cfg['prefix']}{cfg['sep']}{norm_v}"
            row[feat_key] = 1 if norm_val == norm_v else 0

    # 3. 모델이 요구하는 피처 순서로 정렬, 없는 피처는 0 채움
    for feat in MODEL_FEATURES:
        if feat not in row:
            row[feat] = 0

    return pd.DataFrame([row])[MODEL_FEATURES]


# 테스트
if sample:
    X = prepare_features(sample)
    print(f"피처 변환 완료: shape = {X.shape}")
    print(X[[
        "cu_thick_avg", "etch_factor", "meas_etch_temp",
        "meas_etch_sg", "meas_etch_cu", "expo_eq_id_EXP-001"
    ]].to_string(index=False))

## 5. ML 예측 (LightGBM)

In [ ]:
def predict_etch_speed(lot_data: dict) -> tuple[float, pd.DataFrame]:
    """LightGBM으로 에칭 속도 예측
    
    Returns:
        (예측값, 피처 DataFrame)
    """
    X = prepare_features(lot_data)
    pred = lgbm_model.predict(X)[0]
    return float(pred), X


def get_opls_bounds() -> dict:
    """에칭 공정 OPLS 기준값 (하드코딩 — 실제는 MES에서 조회)"""
    return {
        "meas_etch_temp"    : {"lcl": 44.5, "sl": 48.0, "ucl": 53.0, "unit": "°C",  "name": "에칭 온도"},
        "meas_etch_sg"      : {"lcl": 1.32,  "sl": 1.37,  "ucl": 1.42,  "unit": "",    "name": "에칭 비중"},
        "meas_etch_cu"      : {"lcl": 125.0, "sl": 155.0, "ucl": 185.0, "unit": "g/L", "name": "에칭 Cu 농도"},
        "meas_etch_hcl"     : {"lcl": 0.3,   "sl": 0.5,   "ucl": 0.7,   "unit": "N",   "name": "에칭 HCl"},
        "meas_etch_additive": {"lcl": 2.6,   "sl": 3.0,   "ucl": 3.4,   "unit": "g/L", "name": "에칭 첨가제"},
    }


def check_opls_status(X: pd.DataFrame) -> list[dict]:
    """OPLS 기준 대비 이탈 항목 체크"""
    bounds = get_opls_bounds()
    alerts = []
    for feat, lim in bounds.items():
        if feat not in X.columns:
            continue
        val = X[feat].iloc[0]
        if pd.isna(val):
            continue
        status = "정상"
        if val > lim["ucl"]:
            status = "UCL 초과 (상한 이탈)"
        elif val < lim["lcl"]:
            status = "LCL 미달 (하한 이탈)"
        elif val > lim["sl"] * 1.02:
            status = "SL 상향 근접"
        elif val < lim["sl"] * 0.98:
            status = "SL 하향 근접"
        alerts.append({
            "피처"  : feat,
            "항목"  : lim["name"],
            "현재값": round(val, 4),
            "LCL"   : lim["lcl"],
            "SL"    : lim["sl"],
            "UCL"   : lim["ucl"],
            "단위"  : lim["unit"],
            "상태"  : status,
        })
    return alerts


# 테스트
if sample:
    pred_speed, X = predict_etch_speed(sample)
    actual_speed  = sample.get("부식 Speed", "N/A")
    print(f"=== LOT: {sample['LOT']} ===")
    print(f"예측 에칭 속도 : {pred_speed:.4f} m/min")
    print(f"실제 에칭 속도 : {actual_speed} m/min")
    if isinstance(actual_speed, (int, float)):
        err = abs(pred_speed - actual_speed) / actual_speed * 100
        print(f"오차율          : {err:.2f}%")
    
    print("\n=== OPLS 상태 ===" )
    alerts = check_opls_status(X)
    df_alerts = pd.DataFrame(alerts)
    print(df_alerts.to_string(index=False))

## 6. RAG 지식베이스 준비

In [ ]:
def build_knowledge_context(max_rules: int = 5) -> str:
    """RAG 지식베이스를 LLM 시스템 프롬프트용 텍스트로 변환"""
    parts = []

    # 논문 기반 공정 rule (상위 N개)
    if paper_rules:
        parts.append("[논문 기반 공정 규칙]")
        for item in paper_rules[:max_rules]:
            parts.append(json.dumps(item, ensure_ascii=False, indent=2))

    # SHAP 분석 결과
    if shap_content:
        parts.append("\n[SHAP 모델 해석 분석]")
        parts.append(shap_content[:3000])  # 토큰 절약

    return "\n\n".join(parts)


def retrieve_relevant_rules(question: str, lot_data: dict, top_k: int = 3) -> str:
    """키워드 기반 관련 rule 검색 (경량 RAG)"""
    keywords = []
    q_lower  = question.lower()

    # 질문 키워드 추출
    keyword_map = {
        "온도"   : ["온도", "temperature", "temp"],
        "비중"   : ["비중", "density", "sg"],
        "속도"   : ["속도", "speed", "컨베이어"],
        "Cu"     : ["cu", "구리", "copper"],
        "불량"   : ["불량", "defect", "이상", "문제"],
        "수율"   : ["수율", "yield"],
        "SHAP"   : ["shap", "중요도", "영향"],
        "과에칭" : ["과에칭", "over-etch", "선폭"],
        "잔동"   : ["잔동", "under-etch"],
    }
    for key, terms in keyword_map.items():
        if any(t in q_lower for t in terms):
            keywords.append(key)

    # paper_rules에서 키워드 매칭
    scored = []
    for item in paper_rules:
        text  = json.dumps(item, ensure_ascii=False).lower()
        score = sum(1 for kw in keywords if kw.lower() in text)
        if score > 0:
            scored.append((score, item))

    scored.sort(key=lambda x: -x[0])
    top_rules = [json.dumps(item, ensure_ascii=False, indent=2) for _, item in scored[:top_k]]

    if not top_rules and paper_rules:
        top_rules = [json.dumps(paper_rules[0], ensure_ascii=False, indent=2)]

    result = "\n---\n".join(top_rules)
    if shap_content:
        result += "\n\n[SHAP 분석]\n" + shap_content[:2000]
    return result


# 테스트
ctx = retrieve_relevant_rules("에칭 온도가 높을 때 문제가 뭐가 있나요?", {})
print(f"검색된 컨텍스트 길이: {len(ctx)} 자")
print(ctx[:500])

In [ ]:
## 7-a. EXAONE LLM + Hybrid RAG 초기화 (최초 1회 — GPU 필요)
import torch
from transformers import BitsAndBytesConfig
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List as _List

E5_PREFIX = "Instruct: 공정 이상 원인과 조치 방법을 찾으세요\nQuery: "

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. Colab Runtime → 런타임 유형 변경 → GPU 선택 후 재실행하세요.")

# ── 임베딩 모델 ───────────────────────────────────────────────────────────────
print("임베딩 모델 로드 중...")
_emb_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",
    encode_kwargs={"normalize_embeddings": True},
)
print("임베딩 모델 로드 완료")

# ── 지식베이스 → Chroma + BM25 ───────────────────────────────────────────────
_docs = []
for item in paper_rules:
    _docs.append(Document(
        page_content=json.dumps(item, ensure_ascii=False, indent=2),
        metadata={"type": "paper_rule"}
    ))
if shap_content:
    _docs.append(Document(page_content=shap_content, metadata={"type": "shap_analysis"}))

_splitter   = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
_split_docs = _splitter.split_documents(_docs)

_database   = Chroma.from_documents(
    documents=_split_docs,
    embedding=_emb_model,
    collection_name="pragma_rag",
    persist_directory=CHROMA_DIR,
)
_vector_ret = _database.as_retriever(search_kwargs={"k": 5})
_bm25_ret   = BM25Retriever.from_documents(_split_docs)
_bm25_ret.k = 5

class _HybridRetriever(BaseRetriever):
    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> _List[Document]:
        vec_docs  = _vector_ret.invoke(E5_PREFIX + query)
        bm25_docs = _bm25_ret.invoke(query)
        seen, combined = set(), []
        for doc in vec_docs + bm25_docs:
            key = doc.page_content[:80]
            if key not in seen:
                seen.add(key)
                combined.append(doc)
        return combined[:7]

    async def _aget_relevant_documents(self, query: str, **kwargs):
        return self._get_relevant_documents(query)

rag_retriever = _HybridRetriever()
print(f"RAG 구성 완료 (청크 수: {len(_split_docs)}, Chroma 경로: {CHROMA_DIR})")

# ── EXAONE LLM ────────────────────────────────────────────────────────────────
print("EXAONE LLM 로드 중... (최초 1회, 수 분 소요)")
_quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
_pipeline = HuggingFacePipeline.from_model_id(
    model_id="LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 512, "do_sample": False, "repetition_penalty": 1.03},
    model_kwargs={"quantization_config": _quant_cfg, "trust_remote_code": True},
)
exaone_llm = ChatHuggingFace(llm=_pipeline)
print("EXAONE LLM 로드 완료")

## 7. Claude LLM 통합 (RAG + ML 예측 결과 기반 Q&A)

In [ ]:
## 7-b. LOT 요약 + EXAONE Model RAG Q&A 함수

def build_lot_summary(lot_data: dict, pred_speed: float, X: pd.DataFrame, alerts: list) -> str:
    """LOT 공정 현황 요약 문자열 생성"""
    def safe_get(key, digits=4):
        v = lot_data.get(key)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "N/A"
        return round(v, digits) if isinstance(v, float) else v

    alert_lines = []
    for a in alerts:
        if a["상태"] != "정상":
            alert_lines.append(
                f"  ⚠ {a['항목']}: {a['현재값']}{a['단위']} [{a['상태']}] "
                f"(LCL={a['LCL']}, SL={a['SL']}, UCL={a['UCL']})"
            )
    alert_str = "\n".join(alert_lines) if alert_lines else "  이탈 항목 없음"

    etch_cols = {
        "에칭 Cu 농도"  : safe_get(next((c for c in lot_data if "Etching" in c and "Cu" in c  and "분석치" in c), ""), 2),
        "에칭 HCl"      : safe_get(next((c for c in lot_data if "HCl" in c                   and "분석치" in c), ""), 3),
        "에칭 비중"      : safe_get(next((c for c in lot_data if "비중" in c                   and "분석치" in c), ""), 4),
        "에칭 온도"      : safe_get(next((c for c in lot_data if "온도" in c                   and "분석치" in c), ""), 2),
        "에칭 첨가제"    : safe_get(next((c for c in lot_data if "첨가제" in c                 and "분석치" in c), ""), 3),
        "에칭량"         : safe_get(next((c for c in lot_data if "Etching량" in c             and "분석치" in c), ""), 2),
    }
    etch_str = "\n".join(f"  {k}: {v}" for k, v in etch_cols.items())

    return f"""LOT 번호     : {lot_data.get('LOT')}
제품 코드    : {safe_get('통합코드')}
거래처       : {safe_get('거래처')}
공법 구분    : {safe_get('공법구분')}
LAYER        : {safe_get('LAYER')}
DRY FILM     : {safe_get('DRY FILM 정보')}

[에칭 공정 실측치]
{etch_str}
  Cu 표면두께 평균: {safe_get('Cu 표면두께 AVG_VAL')}

[ML 예측 결과]
  예측 에칭 속도 : {pred_speed:.4f} m/min
  실제 에칭 속도 : {safe_get('부식 Speed')} m/min
  검사 결과      : {safe_get('Result Ng2')}

[OPLS 공정 기준 대비 상태]
{alert_str}
"""


MODEL_RAG_TEMPLATE = """다음 문맥을 참고하여 질문에 답변해 주세요.

문맥에는 논문 기반 공정 rule, SHAP 기반 모델 해석 정보,
그리고 현재 LOT의 ML 예측 결과가 포함됩니다.

[현재 공정 실측 및 ML 예측 현황]
{lot_context}

[지식베이스 컨텍스트]
{knowledge_context}

답변 시 아래 내용을 중심으로 정리해 주세요.
- 현재 LOT 상태와 연계한 핵심 답변
- 관련 공정 변수
- 모델/문헌 기반 근거
- 조치 방향

질문:
{question}

답변:
"""


def ask_pragma(
    question  : str,
    lot_data  : dict,
    pred_speed: float,
    X         : pd.DataFrame,
    alerts    : list,
    verbose   : bool = True,
) -> str:
    """EXAONE Model RAG로 LOT 기반 공정 Q&A 수행"""
    lot_summary = build_lot_summary(lot_data, pred_speed, X, alerts)

    docs = rag_retriever.invoke(question)
    knowledge_context = "\n\n".join(
        f"[type={d.metadata.get('type','')}]\n{d.page_content}"
        for d in docs
    )

    prompt_text = MODEL_RAG_TEMPLATE.format(
        lot_context       = lot_summary,
        knowledge_context = knowledge_context,
        question          = question,
    )

    if verbose:
        print(f"[질문]: {question}")
        print("EXAONE 호출 중...")

    answer = exaone_llm.invoke(prompt_text).content

    if verbose:
        print("\n[답변]:")
        print(answer)

    return answer


print("build_lot_summary() + ask_pragma() 정의 완료")

## 8. 통합 실행 함수

In [ ]:
def analyze_lot(lotno: str, question: str) -> str:
    """LOT 번호 + 질문 → EXAONE Model RAG 공정 분석 답변"""
    print(f"\n{'='*60}")
    print(f" LOT {lotno} 분석 시작")
    print(f"{'='*60}")

    # Step 1: MES에서 공정 데이터 조회
    lot_data = get_lot_data(lotno)
    if lot_data is None:
        return f"LOT '{lotno}' 를 찾을 수 없습니다."

    # Step 2: LightGBM으로 에칭 속도 예측
    pred_speed, X = predict_etch_speed(lot_data)
    print(f"▶ 예측 에칭 속도: {pred_speed:.4f} m/min  "
          f"(실제: {lot_data.get('부식 Speed', 'N/A')} m/min)")

    # Step 3: OPLS 기준 이탈 체크
    alerts   = check_opls_status(X)
    abnormal = [a for a in alerts if a["상태"] != "정상"]
    if abnormal:
        print(f"▶ OPLS 이탈 항목: {len(abnormal)}개")
        for a in abnormal:
            print(f"   - {a['항목']}: {a['현재값']}{a['단위']} [{a['상태']}]")
    else:
        print("▶ OPLS 이탈 항목: 없음")

    # Step 4: EXAONE Model RAG로 답변 생성
    print()
    answer = ask_pragma(question, lot_data, pred_speed, X, alerts, verbose=True)
    return answer


print("analyze_lot() 정의 완료")

## 9. 테스트 실행

LOT 번호와 질문을 바꿔가며 실행하세요.

In [ ]:
# ── 여기를 수정하세요 ─────────────────────────────────────────────────────────
LOT_NO   = "A20000"
QUESTION = "이 lot의 에칭 공정 상태를 분석하고, 예측 속도와 실제 속도 차이의 원인 및 개선 방향을 설명해 주세요."
# ─────────────────────────────────────────────────────────────────────────────

answer = analyze_lot(LOT_NO, QUESTION)

In [ ]:
# 다른 질문 예시
LOT_NO2   = "A20001"
QUESTION2 = "에칭 온도와 비중이 OPLS 기준에서 어느 정도 이탈해 있으며, 과에칭 위험도는 어떻게 판단하나요?"

answer2 = analyze_lot(LOT_NO2, QUESTION2)

## 10. 배치 분석 (여러 LOT 한 번에 처리)

In [ ]:
def batch_predict(lot_list: list[str]) -> pd.DataFrame:
    """여러 LOT의 예측 결과 및 OPLS 상태를 DataFrame으로 반환 (LLM 호출 없음)"""
    results = []
    for lotno in lot_list:
        lot_data = get_lot_data(lotno)
        if lot_data is None:
            results.append({"LOT": lotno, "예측속도": None, "실제속도": None, "이탈항목": "NOT FOUND"})
            continue

        pred, X   = predict_etch_speed(lot_data)
        actual    = lot_data.get("부식 Speed")
        alerts    = check_opls_status(X)
        abnormal  = [a["항목"] for a in alerts if a["상태"] != "정상"]

        results.append({
            "LOT"        : lotno,
            "예측속도"   : round(pred, 4),
            "실제속도"   : actual,
            "오차율(%)"  : round(abs(pred - actual) / actual * 100, 2) if isinstance(actual, (int, float)) else None,
            "검사결과"   : lot_data.get("Result Ng2"),
            "OPLS이탈"   : ", ".join(abnormal) if abnormal else "정상",
        })

    return pd.DataFrame(results)


# 처음 10개 LOT 배치 분석
sample_lots = df_mes["LOT"].head(10).tolist()
df_batch = batch_predict(sample_lots)
print(df_batch.to_string(index=False))

## 11. Streamlit 앱 코드 생성

아래 셀을 실행하면 `pragma_streamlit.py` 파일이 생성됩니다.  
`streamlit run pragma_streamlit.py` 로 실행하세요.

## 12. 실시간 공정 모니터링 앱 생성

아래 셀을 실행하면 `pragma_monitor.py`가 생성됩니다.  
`streamlit run pragma_monitor.py` 로 실행하세요.

- **10초 단위** 자동 갱신
- DES 설비 담당 파라미터 시계열 실시간 표시
- Ornstein-Uhlenbeck 확률 과정으로 Train_0319.csv 통계 기반 시뮬레이션
- OPLS 관리 한계선 (LCL / SL / UCL) 표시

In [ ]:
MONITOR_CODE = r'''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import time

# ── 공정 파라미터 설정 (Train_0319.csv 통계 기반) ─────────────────────────────
# Ornstein-Uhlenbeck: dX = θ(μ - X)dt + σ dW
# theta: 평균 회귀 속도 (낮을수록 느리게 복귀 = 더 smooth)
# sigma: 순간 변동 폭 (실제 std의 약 5~10%)

PARAMS = {
    "에칭 온도 (°C)": {
        "mean": 48.00, "theta": 0.08, "sigma": 0.04,
        "lcl": 44.5,  "sl": 48.0,  "ucl": 53.0,
        "fmt": ".2f", "color": "#E74C3C",
    },
    "에칭 비중": {
        "mean": 1.363, "theta": 0.10, "sigma": 0.0015,
        "lcl": 1.32,  "sl": 1.37,  "ucl": 1.42,
        "fmt": ".4f", "color": "#3498DB",
    },
    "에칭 Cu 농도 (g/L)": {
        "mean": 151.4, "theta": 0.06, "sigma": 0.30,
        "lcl": 125.0, "sl": 155.0, "ucl": 185.0,
        "fmt": ".1f", "color": "#2ECC71",
    },
    "에칭 첨가제 (g/L)": {
        "mean": 3.003, "theta": 0.10, "sigma": 0.012,
        "lcl": 2.6,   "sl": 3.0,   "ucl": 3.4,
        "fmt": ".3f", "color": "#9B59B6",
    },
    "현상액 pH": {
        "mean": 11.40, "theta": 0.08, "sigma": 0.015,
        "lcl": 10.8,  "sl": 11.4,  "ucl": 12.0,
        "fmt": ".2f", "color": "#F39C12",
    },
    "현상액 농도": {
        "mean": 1.096, "theta": 0.10, "sigma": 0.004,
        "lcl": 1.00,  "sl": 1.10,  "ucl": 1.20,
        "fmt": ".3f", "color": "#1ABC9C",
    },
}

WINDOW   = 30          # 표시할 데이터 포인트 수 (30 × 10s = 5분)
INTERVAL = 10          # 갱신 주기 (초)

def ou_next(x, cfg):
    """Ornstein-Uhlenbeck 다음 값 (공정 특성 반영 smooth 변화)"""
    drift = cfg["theta"] * (cfg["mean"] - x)
    noise = np.random.normal(0, cfg["sigma"])
    return x + drift + noise

def status_color(val, lcl, ucl):
    if val > ucl or val < lcl:
        return "🔴"
    return "🟢"

# ── Session State 초기화 ──────────────────────────────────────────────────────
if "history" not in st.session_state:
    now = datetime.now()
    ts  = [now - timedelta(seconds=(WINDOW - i) * INTERVAL) for i in range(WINDOW)]
    st.session_state.history = {
        "timestamps": ts,
        **{name: [cfg["mean"] + np.random.normal(0, cfg["sigma"] * 3)
                  for _ in range(WINDOW)]
           for name, cfg in PARAMS.items()},
    }

if "last_update" not in st.session_state:
    st.session_state.last_update = time.time()

# ── 새 데이터 포인트 추가 (10초 경과 시) ─────────────────────────────────────
elapsed = time.time() - st.session_state.last_update
if elapsed >= INTERVAL:
    hist = st.session_state.history
    hist["timestamps"].append(datetime.now())
    hist["timestamps"] = hist["timestamps"][-WINDOW:]
    for name, cfg in PARAMS.items():
        prev = hist[name][-1]
        hist[name].append(ou_next(prev, cfg))
        hist[name] = hist[name][-WINDOW:]
    st.session_state.last_update = time.time()

# ── UI ────────────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="PRAGma 실시간 모니터링",
    page_icon="📡",
    layout="wide",
)
st.title("📡 PRAGma — DES 설비 실시간 공정 모니터링")
st.caption(f"10초 단위 갱신 | 최근 {WINDOW * INTERVAL // 60}분 표시 | "
           f"마지막 갱신: {datetime.now().strftime('%H:%M:%S')}")

# ── 현재값 카드 ───────────────────────────────────────────────────────────────
hist = st.session_state.history
cols = st.columns(len(PARAMS))
for col, (name, cfg) in zip(cols, PARAMS.items()):
    cur = hist[name][-1]
    ico = status_color(cur, cfg["lcl"], cfg["ucl"])
    delta_str = f"{cur - cfg['sl']:+.3f}" if abs(cur - cfg['sl']) > 0.001 else "±0"
    col.metric(
        label=f"{ico} {name}",
        value=format(cur, cfg["fmt"]),
        delta=f"SL 대비 {delta_str}",
        delta_color="off",
    )

st.divider()

# ── 시계열 차트 (2열 3행) ─────────────────────────────────────────────────────
param_list = list(PARAMS.items())
n_rows = (len(param_list) + 1) // 2

fig = make_subplots(
    rows=n_rows, cols=2,
    subplot_titles=[name for name, _ in param_list],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
)

ts_labels = [t.strftime("%H:%M:%S") for t in hist["timestamps"]]

for idx, (name, cfg) in enumerate(param_list):
    row = idx // 2 + 1
    col = idx % 2 + 1
    vals = hist[name]

    # 실측 라인
    fig.add_trace(go.Scatter(
        x=ts_labels, y=vals,
        mode="lines+markers",
        name=name,
        line=dict(color=cfg["color"], width=2),
        marker=dict(size=4),
        showlegend=False,
    ), row=row, col=col)

    # UCL / SL / LCL 기준선
    for lim_val, lim_name, lim_color, lim_dash in [
        (cfg["ucl"], "UCL", "red",   "dash"),
        (cfg["sl"],  "SL",  "gray",  "dot"),
        (cfg["lcl"], "LCL", "blue",  "dash"),
    ]:
        fig.add_hline(
            y=lim_val,
            line=dict(color=lim_color, width=1, dash=lim_dash),
            annotation_text=f"{lim_name}={lim_val}",
            annotation_position="right",
            annotation_font_size=9,
            row=row, col=col,
        )

    # 이탈 구간 배경색
    for i, v in enumerate(vals):
        if v > cfg["ucl"] or v < cfg["lcl"]:
            fig.add_vrect(
                x0=ts_labels[max(0, i - 1)],
                x1=ts_labels[min(len(ts_labels) - 1, i + 1)],
                fillcolor="red", opacity=0.1, line_width=0,
                row=row, col=col,
            )

fig.update_layout(
    height=320 * n_rows,
    margin=dict(l=40, r=80, t=40, b=40),
    paper_bgcolor="white",
    plot_bgcolor="#F8F9FA",
)
fig.update_xaxes(showticklabels=False)   # x축 레이블은 마지막 행만 표시
for i in range(1, n_rows + 1):
    fig.update_xaxes(showticklabels=True, row=i, col=1)
    fig.update_xaxes(showticklabels=True, row=i, col=2)

st.plotly_chart(fig, use_container_width=True)

# ── 자동 갱신 ─────────────────────────────────────────────────────────────────
time.sleep(INTERVAL)
st.rerun()
'''

out_path = BASE_DIR / "pragma_monitor.py"
out_path.write_text(MONITOR_CODE.strip(), encoding="utf-8")
print(f"모니터링 앱 저장 완료: {out_path}")
print(f"\n실행 명령어:")
print(f"  streamlit run {out_path}")

In [ ]:
STREAMLIT_CODE = r'''
import os, json, pickle, time
import numpy as np
import pandas as pd
import streamlit as st
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime, timedelta

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
BASE_DIR = Path(__file__).parent

# ── 리소스 로드 ───────────────────────────────────────────────────────────────
@st.cache_resource
def load_resources():
    with open(BASE_DIR / "notebooks/best_LightGBM_mass_speed_regressor.pkl", "rb") as f:
        model = pickle.load(f)
    df = pd.read_csv(BASE_DIR / "data/Train_0319.csv", encoding="cp949")
    with open(BASE_DIR / "data/rag_data_all.json", "r", encoding="utf-8") as f:
        rules = json.load(f)
    shap_md = BASE_DIR / "notebooks/shap_analysis_for_rag.md"
    shap    = shap_md.read_text(encoding="utf-8") if shap_md.exists() else ""
    return model, df, rules, shap

lgbm_model, df_mes, paper_rules, shap_content = load_resources()
MODEL_FEATURES = lgbm_model.feature_name_

# ── 피처 매핑 ─────────────────────────────────────────────────────────────────
CONTINUOUS_MAP = {
    "Cu 표면두께 Max_Val": "cu_thick_max", "Cu 표면두께 AVG_VAL": "cu_thick_avg",
    "Cu 표면두께 Min_Val": "cu_thick_min", "Cu 표면두께 Std_Val": "cu_thick_std",
    "Cu 표면두께 Median_Val": "cu_thick_median",
}
_sfx = {
    "Etch factor":"etch_factor","Etching(염화동) - Cu":"meas_etch_cu",
    "Etching(염화동) - HCl":"meas_etch_hcl","Etching(염화동) - 비중":"meas_etch_sg",
    "Etching(염화동) - 온도":"meas_etch_temp","Etching-첨가제(HB-120EF)":"meas_etch_additive",
    "Etching량":"meas_etch_amount","Soft Etch - Cu":"meas_softetch_cu",
    "Soft Etch - H2SO4":"meas_softetch_h2so4","Soft Etch - SPS":"meas_softetch_sps",
    "박리액 - 농도":"meas_strip_conc","수세수 - pH":"meas_rinse_ph",
    "현상액 - pH":"meas_dev_ph","현상액 - 농도":"meas_dev_conc",
}
for col in df_mes.columns:
    if "析치" in col or "분析치" in col or "분석치" in col:
        sfx = col.split("_", 1)[-1] if "_" in col else col
        if sfx in _sfx:
            CONTINUOUS_MAP[col] = _sfx[sfx]

CATEGORICAL_COLS = {
    "재작업사유":   {"prefix":"rework_history","values":["Unknown","기타","기판 겹침","두께 미달","딤플","설비 에러"],"sep":"_"},
    "노광 설비정보":{"prefix":"expo_eq_id",    "values":[f"EXP-{i:03d}" for i in range(1,8)],"sep":"_"},
    "DES 설비정보": {"prefix":"des_eq_id",     "values":[f"DES-{i:03d}" for i in range(1,7)],"sep":"_"},
    "정면 설비정보":{"prefix":"brush_eq_id",   "values":[f"PRE-{i:03d}" for i in range(1,8)],"sep":"_"},
}
OPLS_BOUNDS = {
    "meas_etch_temp":    {"lcl":44.5,"sl":48.0,"ucl":53.0,"unit":"°C", "name":"에칭 온도"},
    "meas_etch_sg":      {"lcl":1.32,"sl":1.37,"ucl":1.42,"unit":"",   "name":"에칭 비중"},
    "meas_etch_cu":      {"lcl":125.,"sl":155.,"ucl":185.,"unit":"g/L","name":"에칭 Cu 농도"},
    "meas_etch_hcl":     {"lcl":0.3, "sl":0.5, "ucl":0.7, "unit":"N",  "name":"에칭 HCl"},
    "meas_etch_additive":{"lcl":2.6, "sl":3.0, "ucl":3.4, "unit":"g/L","name":"에칭 첨가제"},
}

def prepare_features(lot_data):
    row = {}
    for csv_col, feat in CONTINUOUS_MAP.items():
        v = lot_data.get(csv_col, np.nan)
        row[feat] = float(v) if v is not None and str(v) not in ("","nan","None") else np.nan
    for csv_col, cfg in CATEGORICAL_COLS.items():
        raw  = str(lot_data.get(csv_col,"")).strip()
        norm = raw.replace(" ","_") if cfg["prefix"]=="rework_history" else raw
        for v in cfg["values"]:
            nv = v.replace(" ","_") if cfg["prefix"]=="rework_history" else v
            row[f"{cfg['prefix']}{cfg['sep']}{nv}"] = 1 if norm==nv else 0
    for feat in MODEL_FEATURES:
        if feat not in row: row[feat] = 0
    return pd.DataFrame([row])[MODEL_FEATURES]

def check_opls(X):
    alerts = []
    for feat, lim in OPLS_BOUNDS.items():
        if feat not in X.columns: continue
        val = X[feat].iloc[0]
        if pd.isna(val): continue
        status = "UCL 초과" if val>lim["ucl"] else ("LCL 미달" if val<lim["lcl"] else "정상")
        alerts.append({"항목":lim["name"],"현재값":round(val,4),
                        "LCL":lim["lcl"],"SL":lim["sl"],"UCL":lim["ucl"],
                        "단위":lim["unit"],"상태":status})
    return alerts

def ask_exaone_or_simple(question, lot_data, pred, X, alerts):
    """LLM 없이 규칙 기반 간이 답변 (Streamlit 독립 실행용)"""
    abnormal = [a for a in alerts if a["상태"] != "정상"]
    lines = [f"**LOT {lot_data.get('LOT')} 분석 결과**\n"]
    lines.append(f"- 예측 에칭 속도: **{pred:.4f} m/min** (실제: {lot_data.get('부식 Speed','N/A')} m/min)")
    if abnormal:
        lines.append(f"\n**⚠ OPLS 이탈 항목 ({len(abnormal)}개)**")
        for a in abnormal:
            lines.append(f"  - {a['항목']}: {a['현재값']}{a['단위']} [{a['상태']}]  "
                         f"(LCL={a['LCL']}, SL={a['SL']}, UCL={a['UCL']})")
    else:
        lines.append("\n**✅ 모든 공정 파라미터 정상 범위**")
    lines.append(f"\n*질문: {question}*")
    lines.append("\n> EXAONE LLM 연동 시 pragma_app.ipynb의 analyze_lot()을 사용하세요.")
    return "\n".join(lines)

# ═══════════════════════════════════════════════════════════════════════════════
# 실시간 모니터링 설정 (Train_0319.csv 통계 기반 OU 프로세스)
# ═══════════════════════════════════════════════════════════════════════════════
MON_PARAMS = {
    "에칭 온도 (°C)":   {"mean":48.00,"theta":0.08,"sigma":0.04, "lcl":44.5,"sl":48.0,"ucl":53.0, "fmt":".2f","color":"#E74C3C"},
    "에칭 비중":         {"mean":1.363,"theta":0.10,"sigma":0.0015,"lcl":1.32,"sl":1.37,"ucl":1.42,"fmt":".4f","color":"#3498DB"},
    "에칭 Cu 농도 (g/L)":{"mean":151.4,"theta":0.06,"sigma":0.30, "lcl":125.,"sl":155.,"ucl":185.,"fmt":".1f","color":"#2ECC71"},
    "에칭 첨가제 (g/L)": {"mean":3.003,"theta":0.10,"sigma":0.012,"lcl":2.6, "sl":3.0, "ucl":3.4, "fmt":".3f","color":"#9B59B6"},
    "현상액 pH":         {"mean":11.40,"theta":0.08,"sigma":0.015,"lcl":10.8,"sl":11.4,"ucl":12.0,"fmt":".2f","color":"#F39C12"},
    "현상액 농도":       {"mean":1.096,"theta":0.10,"sigma":0.004,"lcl":1.00,"sl":1.10,"ucl":1.20,"fmt":".3f","color":"#1ABC9C"},
}
MON_WINDOW   = 30    # 최근 30포인트 = 5분
MON_INTERVAL = 10    # 10초 갱신

def ou_next(x, cfg):
    return x + cfg["theta"] * (cfg["mean"] - x) + np.random.normal(0, cfg["sigma"])

def init_history():
    now = datetime.now()
    ts  = [now - timedelta(seconds=(MON_WINDOW - i) * MON_INTERVAL) for i in range(MON_WINDOW)]
    return {
        "timestamps": ts,
        **{name: [cfg["mean"] + np.random.normal(0, cfg["sigma"]*3) for _ in range(MON_WINDOW)]
           for name, cfg in MON_PARAMS.items()},
    }

# ═══════════════════════════════════════════════════════════════════════════════
# Streamlit 앱
# ═══════════════════════════════════════════════════════════════════════════════
st.set_page_config(page_title="PRAGma", page_icon="⚙️", layout="wide")
st.title("⚙️ PRAGma — 에칭 공정 분석 시스템")

tab_lot, tab_mon = st.tabs(["🔍 LOT 분석", "📡 실시간 모니터링"])

# ──────────────────────────────────────────────────────────────────────────────
# TAB 1: LOT 분석
# ──────────────────────────────────────────────────────────────────────────────
with tab_lot:
    col1, col2 = st.columns([1, 2])

    with col1:
        lotno    = st.text_input("LOT 번호", value="A20000", placeholder="예: A20000")
        question = st.text_area("질문", value="이 lot의 에칭 공정 상태를 분석하고 개선 방향을 알려주세요.", height=100)
        run_btn  = st.button("분석 시작", type="primary", use_container_width=True)
        st.caption("EXAONE LLM 답변은 pragma_app.ipynb에서 실행하세요.")

    if run_btn:
        rows = df_mes[df_mes["LOT"] == lotno]
        if rows.empty:
            st.error(f"LOT '{lotno}' 를 찾을 수 없습니다.")
        else:
            lot_data = rows.iloc[0].to_dict()
            X        = prepare_features(lot_data)
            pred     = float(lgbm_model.predict(X)[0])
            actual   = lot_data.get("부식 Speed")
            alerts   = check_opls(X)

            with col2:
                st.subheader(f"LOT {lotno} 분석 결과")
                m1, m2, m3 = st.columns(3)
                m1.metric("예측 에칭 속도", f"{pred:.4f} m/min")
                m2.metric("실제 에칭 속도", f"{actual} m/min",
                          delta=f"{pred-actual:+.4f}" if isinstance(actual,(int,float)) else None)
                m3.metric("검사 결과", lot_data.get("Result Ng2","N/A"))

                st.divider()
                st.subheader("OPLS 공정 기준 상태")
                df_al = pd.DataFrame(alerts)
                def _color(v):
                    return "background-color:#ffcccc" if v!="정상" else "background-color:#ccffcc"
                st.dataframe(df_al.style.map(_color, subset=["상태"]), use_container_width=True)

                st.divider()
                st.subheader("AI 공정 분석")
                with st.spinner("분석 중..."):
                    answer = ask_exaone_or_simple(question, lot_data, pred, X, alerts)
                st.markdown(answer)

# ──────────────────────────────────────────────────────────────────────────────
# TAB 2: 실시간 모니터링
# ──────────────────────────────────────────────────────────────────────────────
with tab_mon:
    st.subheader("📡 DES 설비 공정 파라미터 실시간 모니터링")
    st.caption(f"10초 단위 갱신 | 최근 {MON_WINDOW * MON_INTERVAL // 60}분 표시 | "
               f"Ornstein-Uhlenbeck 시뮬레이션 (Train_0319.csv 통계 기반)")

    # Session State 초기화
    if "mon_history" not in st.session_state:
        st.session_state.mon_history = init_history()
    if "mon_last" not in st.session_state:
        st.session_state.mon_last = time.time()

    # 10초 경과 시 새 포인트 추가
    if time.time() - st.session_state.mon_last >= MON_INTERVAL:
        hist = st.session_state.mon_history
        hist["timestamps"].append(datetime.now())
        hist["timestamps"] = hist["timestamps"][-MON_WINDOW:]
        for name, cfg in MON_PARAMS.items():
            hist[name].append(ou_next(hist[name][-1], cfg))
            hist[name] = hist[name][-MON_WINDOW:]
        st.session_state.mon_last = time.time()

    hist = st.session_state.mon_history

    # 현재값 카드
    metric_cols = st.columns(len(MON_PARAMS))
    for mcol, (name, cfg) in zip(metric_cols, MON_PARAMS.items()):
        cur = hist[name][-1]
        ico = "🔴" if cur > cfg["ucl"] or cur < cfg["lcl"] else "🟢"
        mcol.metric(f"{ico} {name}", format(cur, cfg["fmt"]),
                    delta=f"SL 대비 {cur - cfg['sl']:+.3f}", delta_color="off")

    st.divider()

    # 시계열 차트
    param_list = list(MON_PARAMS.items())
    n_rows = (len(param_list) + 1) // 2
    fig = make_subplots(rows=n_rows, cols=2,
                        subplot_titles=[n for n, _ in param_list],
                        vertical_spacing=0.12, horizontal_spacing=0.08)
    ts_labels = [t.strftime("%H:%M:%S") for t in hist["timestamps"]]

    for idx, (name, cfg) in enumerate(param_list):
        r, c = idx // 2 + 1, idx % 2 + 1
        vals = hist[name]
        fig.add_trace(go.Scatter(x=ts_labels, y=vals, mode="lines+markers",
                                 line=dict(color=cfg["color"], width=2),
                                 marker=dict(size=4), showlegend=False), row=r, col=c)
        for lv, ln, lc, ld in [(cfg["ucl"],"UCL","red","dash"),
                                (cfg["sl"], "SL", "gray","dot"),
                                (cfg["lcl"],"LCL","blue","dash")]:
            fig.add_hline(y=lv, line=dict(color=lc, width=1, dash=ld),
                          annotation_text=f"{ln}={lv}", annotation_position="right",
                          annotation_font_size=9, row=r, col=c)
        for i, v in enumerate(vals):
            if v > cfg["ucl"] or v < cfg["lcl"]:
                fig.add_vrect(x0=ts_labels[max(0,i-1)], x1=ts_labels[min(len(ts_labels)-1,i+1)],
                              fillcolor="red", opacity=0.1, line_width=0, row=r, col=c)

    fig.update_layout(height=300*n_rows, margin=dict(l=40,r=80,t=40,b=40),
                      paper_bgcolor="white", plot_bgcolor="#F8F9FA")
    st.plotly_chart(fig, use_container_width=True)

    # 자동 갱신 버튼
    if st.button("🔄 지금 갱신", key="refresh_btn"):
        st.rerun()
    st.caption(f"마지막 갱신: {datetime.now().strftime('%H:%M:%S')} | 10초 후 자동 갱신됩니다.")
    time.sleep(MON_INTERVAL)
    st.rerun()
'''

out_path = BASE_DIR / "pragma_streamlit.py"
out_path.write_text(STREAMLIT_CODE.strip(), encoding="utf-8")
print(f"Streamlit 앱 저장 완료: {out_path}")
print(f"\n실행 명령어:")
print(f"  streamlit run {out_path}")

## 13. Colab에서 Streamlit 실행 (ngrok 터널링)

In [ ]:
import subprocess, time
from pyngrok import ngrok

# pyngrok 설치 (없을 경우)
subprocess.run(["pip", "install", "-q", "pyngrok", "streamlit", "plotly"], check=True)

# 기존 프로세스 정리
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(1)

# Streamlit 백그라운드 실행
proc = subprocess.Popen(
    ["streamlit", "run", str(BASE_DIR / "pragma_streamlit.py"),
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)  # 기동 대기

# ngrok 터널 오픈
public_url = ngrok.connect(8501)
print("=" * 50)
print(f"  PRAGma 앱 주소: {public_url}")
print("=" * 50)
print("브라우저에서 위 주소를 열어주세요.")
print("종료하려면: proc.terminate()")
